In [2]:
import pandas as pd
import GEOparse

print("pandas", pd.__version__)

pandas 3.1.1


In [3]:
from pathlib import Path

raw_dir = Path("../data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

gse = GEOparse.get_GEO(geo="GSE98793", destdir=str(raw_dir))

10-Nov-2026 19:06:28 DEBUG utils - Directory ../data/raw already exists. Skipping.
10-Nov-2026 19:06:28 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE98nnn/GSE98793/soft/GSE98793_family.soft.gz to ../data/raw/GSE98793_family.soft.gz
100%|██████████| 104M/104M [00:27<00:00, 3.94MB/s]    
10-Nov-2026 19:07:04 DEBUG downloader - Size validation passed
10-Nov-2026 19:07:04 DEBUG downloader - Moving /var/folders/gl/b1gprnk51j707yyn_q18xgqh0000gn/T/tmp2wlwbbv5 to /Users/claudia/Documents/GitHub/inflammatory-depression-transcriptome/data/raw/GSE98793_family.soft.gz
10-Nov-2026 19:07:04 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE98nnn/GSE98793/soft/GSE98793_family.soft.gz
10-Nov-2026 19:07:04 INFO GEOparse - Parsing ../data/raw/GSE98793_family.soft.gz: 
10-Nov-2026 19:07:04 DEBUG GEOparse - DATABASE: GeoMiame
10-Nov-2026 19:07:04 DEBUG GEOparse - SERIES: GSE98793
10-Nov-2026 19:07:04 DEBUG GEOparse - PLATFORM: GPL570
/Users/claudi

In [4]:
pheno = gse.phenotype_data
print(pheno.shape)
pheno.columns.tolist()

(192, 37)


['title',
 'geo_accession',
 'status',
 'submission_date',
 'last_update_date',
 'type',
 'channel_count',
 'source_name_ch1',
 'organism_ch1',
 'taxid_ch1',
 'characteristics_ch1.0.subject group',
 'characteristics_ch1.1.anxiety',
 'characteristics_ch1.2.gender',
 'characteristics_ch1.3.age',
 'characteristics_ch1.4.tissue',
 'characteristics_ch1.5.batch',
 'molecule_ch1',
 'extract_protocol_ch1',
 'label_ch1',
 'label_protocol_ch1',
 'hyb_protocol',
 'scan_protocol',
 'description',
 'data_processing',
 'platform_id',
 'contact_name',
 'contact_email',
 'contact_laboratory',
 'contact_department',
 'contact_institute',
 'contact_address',
 'contact_city',
 'contact_zip/postal_code',
 'contact_country',
 'supplementary_file',
 'series_id',
 'data_row_count']

In [6]:
columns = {
    "characteristics_ch1.0.subject group": "group",
    "characteristics_ch1.1.anxiety": "anxiety",
    "characteristics_ch1.2.gender": "sex",
    "characteristics_ch1.3.age": "age",
    "characteristics_ch1.4.tissue": "tissue",
    "characteristics_ch1.5.batch": "batch",
}

samples = pheno[list(columns)].rename(columns=columns)
samples.head()

,group,anxiety,sex,age,tissue,batch
GSM2612096,CNTL; healthy control,no,M,35.8,whole blood,1
GSM2612097,CNTL; healthy control,no,F,36.9,whole blood,1
GSM2612098,CNTL; healthy control,no,F,62,whole blood,1
GSM2612099,CNTL; healthy control,no,F,35.7,whole blood,1
GSM2612100,CNTL; healthy control,no,M,53.4,whole blood,1


In [8]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

samples.to_csv(processed_dir / "GSE98793_samples.csv")

In [9]:
for column in ["group", "anxiety", "sex", "tissue", "batch"]:
    print(samples[column].value_counts(), "\n")

pd.crosstab(samples["batch"], samples["group"])


group
CASE; major depressive disorder (MDD) patient    128
CNTL; healthy control                             64
Name: count, dtype: int64 

anxiety
no     128
yes     64
Name: count, dtype: int64 

sex
F    144
M     48
Name: count, dtype: int64 

tissue
whole blood    192
Name: count, dtype: int64 

batch
1    96
2    96
Name: count, dtype: int64 



group,CASE; major depressive disorder (MDD) patient,CNTL; healthy control
batch,,
1,64,32
2,64,32


In [10]:
print(pd.crosstab(samples["anxiety"], samples["group"]), "\n")

samples["age"] = pd.to_numeric(samples["age"])
samples.groupby("group")["age"].describe()

group    CASE; major depressive disorder (MDD) patient  CNTL; healthy control
anxiety                                                                      
no                                                  64                     64
yes                                                 64                      0 



,count,mean,std,min,25%,50%,75%,max
group,,,,,,,,
CASE; major depressive disorder (MDD) patient,128.0,52.039844,11.507049,31.0,43.775,52.65,62.05,72.8
CNTL; healthy control,64.0,52.029688,11.410420,31.2,44.050,52.55,62.05,73.1
